In [ ]:
import os
SAVE_FIGS = os.getenv("PHYTOMNI_SAVE") == "1"
if SAVE_FIGS:
    os.makedirs("output", exist_ok=True)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as offline

In [ ]:
offline.init_notebook_mode(connected=True)
pio.kaleido.scope.default_format = 'pdf'

### Fig. 2a

In [ ]:
MODEL_ORDER = [
    "Phyto-Reasoner",
    "Phyto-Chatbot",
    "GPT-5",
    "o3",
    "Gemini-2.5-Pro",
    "Claude-Opus-4.1",
    "Grok-3-Beta",
    "Deepseek-V3",
    "Deepseek-R1",
]
knowledge_frame = pd.read_csv(
    "PhytoBench-Knowledge-for_plot.tsv",
    sep="\t",
)
required_knowledge_columns = {
    "Model",
    "DisplayLabel",
    "IdentificationAccuracy",
    "IdentificationN",
    "TraceBLEU4",
    "TraceN",
}
if not required_knowledge_columns <= set(knowledge_frame.columns):
    raise ValueError("Fig. 2a input has an invalid schema.")
if knowledge_frame["Model"].tolist() != MODEL_ORDER:
    raise ValueError("Fig. 2a input has an invalid model order.")

model_list = knowledge_frame["DisplayLabel"].tolist()
identification_accuracy = knowledge_frame[
    "IdentificationAccuracy"
].tolist()
trace_bleu4 = knowledge_frame["TraceBLEU4"].tolist()
identification_colors = [
    "rgb(0,102,204)" if model.startswith("Phyto-") else "rgb(128,128,128)"
    for model in knowledge_frame["Model"]
]
trace_colors = [
    "rgb(102,163,255)" if model.startswith("Phyto-") else "rgb(179,179,179)"
    for model in knowledge_frame["Model"]
]

In [ ]:
line_width = 2

fig = go.Figure()

fig.add_trace(go.Bar(
    x=model_list,
    y=[value * 100 - 20 for value in identification_accuracy],
    marker_color=identification_colors,
    marker_line_color='rgb(0,0,0)',
    marker_line_width=line_width,
    textposition='outside',
))
fig.add_trace(go.Bar(
    x=model_list,
    y=-np.array(trace_bleu4) * 600,
    marker_color=trace_colors,
    marker_line_color='rgb(0,0,0)',
    marker_line_width=line_width,
    textposition='outside',
))

fig.update_layout(
    barmode='relative',
    showlegend=False,
    xaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width},
    yaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width, 'title_text': 'Accuracy rate (%)', 'range': [-60, 60]},
    plot_bgcolor='white',
    font_family='Arial',
    font_color='rgb(0,0,0)',
    font_size=20,
    width=1.9598*400,
    height=5.2761*400)

fig.show()
if SAVE_FIGS:
    file_prefix = f'fig.2a.phytobench-knowledge.bar'.lower().replace(' ', '_')
    fig.write_image(f"output/{file_prefix}.pdf")
    fig.write_image(f"output/{file_prefix}.png")

### Fig. 2b

In [ ]:
data_frame = pd.read_csv(
    "PhytoBench-Data-for_plot.tsv",
    sep="\t",
)
required_data_columns = {"Model", "DisplayLabel", "Accuracy", "N"}
if not required_data_columns <= set(data_frame.columns):
    raise ValueError("Fig. 2b input has an invalid schema.")
if data_frame["Model"].tolist() != MODEL_ORDER:
    raise ValueError("Fig. 2b input has an invalid model order.")

model_list = data_frame["DisplayLabel"].tolist()
data_accuracy = data_frame["Accuracy"].tolist()
data_colors = [
    "rgb(31,113,179)" if model.startswith("Phyto-") else "rgb(208,210,211)"
    for model in data_frame["Model"]
]

In [ ]:
line_width = 2

fig = go.Figure()

fig.add_trace(go.Bar(
    x=model_list,
    y=[accuracy * 100 for accuracy in data_accuracy],
    marker_color=data_colors,
    marker_line_color='rgb(0,0,0)',
    marker_line_width=line_width,
    textposition='outside',
    ),
)

fig.update_layout(
    showlegend=False,
    xaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width},
    yaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width, 'title_text': 'Accuracy rate (%)', 'range': [40, 90]},
    plot_bgcolor='white',
    font_family='Arial',
    font_color='rgb(0,0,0)',
    font_size=20,
    width=5.2761*400,
    height=1.9598*400)

fig.show()
if SAVE_FIGS:
    file_prefix = f'fig.2b.phytobench-data.bar'.lower().replace(' ', '_')
    fig.write_image(f"output/{file_prefix}.pdf")
    fig.write_image(f"output/{file_prefix}.png")

### Fig. 2c

In [ ]:
analysis_frame = pd.read_csv(
    "PhytoBench-Analysis-for_plot.tsv",
    sep="\t",
)
required_analysis_columns = {
    "Task",
    "Species",
    "Rep",
    "ObservationID",
    "Model",
    "DisplayLabel",
    "PlanScore",
    "ToolScore",
    "ParameterScore",
    "RateScore",
    "TotalScore",
}
if not required_analysis_columns <= set(analysis_frame.columns):
    raise ValueError("Fig. 2c input has an invalid schema.")
if analysis_frame["Model"].drop_duplicates().tolist() != MODEL_ORDER:
    raise ValueError("Fig. 2c input has an invalid model order.")
analysis_counts = analysis_frame.groupby("Model", sort=False).size()
if analysis_counts.tolist() != [50] * len(MODEL_ORDER):
    raise ValueError("Fig. 2c requires 50 primary benchmark runs per model.")
if not analysis_frame["Species"].eq("Oryza_sativa").all():
    raise ValueError("Fig. 2c must exclude cross-species extension runs.")
if analysis_frame["Task"].nunique() != 10:
    raise ValueError("Fig. 2c requires all 10 analysis scenarios.")
if not analysis_frame.groupby(["Model", "Task"]).size().eq(5).all():
    raise ValueError("Fig. 2c requires five runs per model and scenario.")
component_columns = [
    "PlanScore",
    "ToolScore",
    "ParameterScore",
    "RateScore",
]
if not analysis_frame[component_columns].apply(
    lambda column: column.between(0, 25)
).all().all():
    raise ValueError("Fig. 2c component scores must be within 0–25.")
if not analysis_frame["TotalScore"].between(0, 100).all():
    raise ValueError("Fig. 2c total scores must be within 0–100.")
if not np.allclose(
    analysis_frame[component_columns].sum(axis=1),
    analysis_frame["TotalScore"],
):
    raise ValueError("Fig. 2c total scores must equal the four components.")

In [ ]:
line_width = 2

fig = go.Figure()

for model in MODEL_ORDER:
    model_frame = analysis_frame.loc[analysis_frame["Model"] == model]
    display_label = model_frame["DisplayLabel"].iloc[0]
    fig.add_trace(go.Violin(
        x=[display_label] * len(model_frame),
        y=model_frame["TotalScore"],
        name=display_label,
        line_color='rgb(0,0,0)',
        fillcolor=(
            'rgb(31,113,179)'
            if model.startswith('Phyto-')
            else 'rgb(208,210,211)'
        ),
        box_visible=True,
        meanline_visible=True,
    ))

fig.update_layout(
    showlegend=False,
    xaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width},
    yaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width, 'title_text': 'Total score (0–100)', 'range': [0, 100]},
    plot_bgcolor='white',
    font_family='Arial',
    font_color='rgb(0,0,0)',
    font_size=20,
    width=2100,
    height=784)

fig.show()
if SAVE_FIGS:
    file_prefix = f'fig.2c.phytobench-analysis.violin'.lower().replace(' ', '_')
    fig.write_image(f"output/{file_prefix}.pdf")
    fig.write_image(f"output/{file_prefix}.png")

### Fig. 2d-f

Generated from the frozen five-model ranking aggregates by
`Supplementary Fig. 12-15/supplementary_fig. 12-15.ipynb`.

### Fig. 2g

BERTScore precision is loaded from a dedicated figure-input TSV.

In [ ]:
bertscore_frame = pd.read_csv(
    "PhytoBench-Gene-BERTScore-for_plot.tsv",
    sep="\t",
)
required_bertscore_columns = {"Model", "DisplayLabel", "BERTScorePrecision"}
if not required_bertscore_columns <= set(bertscore_frame.columns):
    raise ValueError("Fig. 2g BERTScore input has an invalid schema.")

model_list = bertscore_frame["DisplayLabel"].tolist()
y_list = bertscore_frame["BERTScorePrecision"].tolist()
color_list = [
    "rgb(31,113,179)" if model == "Phytomni" else "rgb(208,210,211)"
    for model in bertscore_frame["Model"]
]

In [ ]:
line_width = 2

fig = go.Figure()

fig.add_trace(go.Bar(
    x=model_list,
    y=y_list,
    marker_color=color_list,
    marker_line_color='rgb(0,0,0)',
    marker_line_width=line_width,
    textposition='outside',
))

fig.update_layout(
    xaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width, 'tickangle': -15, 'automargin': True},
    yaxis={'showline': True, 'linewidth': line_width, 'linecolor': 'rgb(0,0,0)', 'mirror': True, 'ticks': 'outside', 'tickwidth': line_width, 'title_text': 'BERTScore precision', 'range': [0.47, 0.58]},
    plot_bgcolor='white',
    font_family='Arial',
    font_color='rgb(0,0,0)',
    font_size=20,
    width=1080,
    height=1080)

fig.show()
if SAVE_FIGS:
    file_prefix = f'fig.2g.phytobench-gene.well_studied.bar'.lower().replace(' ', '_')
    fig.write_image(f"output/{file_prefix}.pdf")
    fig.write_image(f"output/{file_prefix}.png")

### Fig. 2h

Hallucination rate is loaded from the dedicated five-model figure-input TSV.

In [ ]:
HALLUCINATION_DATA = Path(
    "PhytoBench-Gene-hallucination-for_plot.tsv"
)
hallucination_frame = pd.read_csv(HALLUCINATION_DATA, sep="\t")
required_hallucination_columns = {
    "Model",
    "DisplayLabel",
    "MeanDirectionalContradictionRatio",
}
if not required_hallucination_columns <= set(hallucination_frame.columns):
    raise ValueError("Fig. 2h hallucination input has an invalid schema.")

In [ ]:
if hallucination_frame is not None:
    model_list = hallucination_frame["DisplayLabel"].tolist()
    y_list = hallucination_frame[
        "MeanDirectionalContradictionRatio"
    ].tolist()
    color_list = [
        "rgb(31,113,179)"
        if model == "Phytomni"
        else "rgb(208,210,211)"
        for model in hallucination_frame["Model"]
    ]
    line_width = 2

    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=model_list,
            y=y_list,
            marker_color=color_list,
            marker_line_color="rgb(0,0,0)",
            marker_line_width=line_width,
            textposition="outside",
        )
    )
    fig.update_layout(
        xaxis={
            "showline": True,
            "linewidth": line_width,
            "linecolor": "rgb(0,0,0)",
            "mirror": True,
            "ticks": "outside",
            "tickwidth": line_width,
        },
        yaxis={
            "showline": True,
            "linewidth": line_width,
            "linecolor": "rgb(0,0,0)",
            "mirror": True,
            "ticks": "outside",
            "tickwidth": line_width,
            "title_text": "Confabulation rate",
            "range": [0.1, 0.7],
        },
        plot_bgcolor="white",
        font_family="Arial",
        font_color="rgb(0,0,0)",
        font_size=20,
        width=1080,
        height=1080,
    )
    fig.show()
    if SAVE_FIGS:
        file_prefix = (
            "fig.2h.phytobench-gene.uncharacterized.bar"
            .lower()
            .replace(" ", "_")
        )
        fig.write_image(f"output/{file_prefix}.pdf")
        fig.write_image(f"output/{file_prefix}.png")